In [4]:
import pandas as pd
import numpy as np
from numpy.linalg import norm
import tensorflow as tf 
import ast 

import os
import plotly.express as px

In [10]:
import sys
sys.path.append(r'D:\dev work\recommender systems\Atrad_CARS\code\v6_fixed_port_size')

In [6]:
max_port_size = 50

# Training Data

In [7]:
retriever_location_ = r"D:\dev work\recommender systems\Atrad_CARS\model_weights\2024_07_01_31\retriever_v3_port_v2__fixed_max_port_size_50"
stock_info_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\stock_data.xlsx"

train_ds_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\retriver_train".format(max_port_size)
test_ds_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\retriver_test".format(max_port_size)
portfolios_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\portfolios".format(max_port_size)


In [8]:
test_ds = tf.data.Dataset.load(test_ds_loc).cache()

train_ds = tf.data.Dataset.load(train_ds_loc).cache()

portfolio_ds = tf.data.Dataset.load(portfolios_loc).cache()

In [11]:
from retrieval_recommender_v2 import Retriever

retriever = Retriever(
    use_timestamp = True,
    portfolios = portfolio_ds
)

retriever.load_weights(retriever_location_)

retriever.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))


In [12]:
stock_info = pd.read_excel(stock_info_loc)
stock_info = stock_info.drop(['Unnamed: 0','buisnesssummary'],axis = 1)
stock_info = stock_info.rename(columns = {
    'symbol':'STOCKCODE',
    'name' : 'STOCKNAME',
    'gics_code' : 'GICS'
})
stock_info = stock_info[~stock_info['GICS'].isna()]

stock_info.shape
print("items data shape :: {}".format(stock_info.shape))
unique_items_ = np.unique(np.concatenate(list(train_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator())))
stock_info = stock_info[stock_info['STOCKCODE'].isin([item.decode('utf-8') for item in unique_items_])]

items_ds = tf.data.Dataset.from_tensor_slices(stock_info.to_dict(orient= 'list'))

items data shape :: (280, 3)


In [13]:
item_ids = np.concatenate(list(items_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator()))

item_vocabulary = dict(zip(item_ids.tolist(), range(len(item_ids))))
item_vocabulary_inv = {v: k for k, v in item_vocabulary.items()}

item_embeddings = np.concatenate(list(items_ds.batch(len(items_ds)).map(lambda x: retriever.item_model(x)).as_numpy_iterator()))

In [14]:
# item_vocabulary

In [15]:
item_embeddings[0]

array([ 0.20670696,  0.0299624 , -0.00841029,  0.21108952,  0.6146847 ,
       -0.05714659,  0.5645191 ,  0.1413633 , -0.39156005,  0.04039523,
        0.27523082,  0.09472056,  0.9829434 , -0.27703106, -0.5640763 ,
       -0.6359535 , -0.02794771, -0.251439  , -0.3716036 ,  0.62373966,
       -0.33609334, -0.14019713, -0.8944054 ,  0.4868043 ,  0.46584725,
       -0.8995875 , -1.1241951 ,  0.10092983,  0.31128347, -0.55777043,
       -1.0755655 ,  0.67482245], dtype=float32)

In [16]:
def cosine_sim(vec1, vec2):

    return np.dot(vec1, vec2)/(norm(vec1)*norm(vec2))

In [17]:
cosine_sim(
    item_embeddings[0],
    item_embeddings[200]
)

0.013481452

In [18]:
train_df = pd.DataFrame(
    data = list(train_ds.as_numpy_iterator())
)

train_df = train_df.astype(
    {
        'CDSACCNO' : 'str',
        'USER_ID' : 'str',
        'STOCKCODE' : 'str',
        'STOCKNAME' : 'str',
        'GICS' : 'str'
        })

In [19]:
train_df['CDSACCNO'].value_counts()

CDSACCNO
CAS-551970108-VN/00    45
COM-80729-LI/00        45
BMS-53923-LI/00        45
COM-788042957-VN/00    45
COM-790042166-VN/00    45
                       ..
BMS-930980498-VN/00    15
COM-51013-LC/00        15
BMS-26913-LI/00        15
BMS-40691-LI/00        15
BMS-850201684-VN/00    15
Name: count, Length: 2984, dtype: int64

In [20]:
test_df = pd.DataFrame(
    data = list(test_ds.as_numpy_iterator())
)

test_df = test_df.astype(
    {
        'CDSACCNO' : 'str',
        'USER_ID' : 'str',
        'STOCKCODE' : 'str',
        'STOCKNAME' : 'str',
        'GICS' : 'str'
        })

In [21]:
portfolio_df = pd.DataFrame(
    data = list(portfolio_ds.as_numpy_iterator())
)

portfolio_df = portfolio_df.astype(
    {
        'CDSACCNO' : 'str',
        'USER_ID' : 'str',
        'STOCKCODE' : 'str',
        'STOCKNAME' : 'str',
        'GICS' : 'str'
        })

In [22]:
unique_train_items = set(train_df['STOCKCODE'].unique())

unique_test_items = set(test_df['STOCKCODE'].unique())

unique_port_items = set(portfolio_df['STOCKCODE'].unique())

In [23]:
len(unique_train_items) , len(unique_test_items) , len(unique_port_items)

(268, 266, 268)

In [24]:
train_df.head()

,GICS,UNIX_TS,STOCKCODE,RATING,STOCKNAME,USER_ID,CDSACCNO
0,Utilities,1.641148e+09,VONE,3.0,VALLIBEL ONE PLC,"[b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'...",BMS-10544-LC/00
1,Food Beverage & Tobacco,1.641148e+09,ELPL,3.0,ELPITIYA PLANTATIONS PLC,"[b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'...",BMS-10544-LC/00
2,Diversified Financials,1.641407e+09,LFIN,2.0,LB FINANCE PLC,"[b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'...",BMS-10544-LC/00
3,Materials,1.641753e+09,TKYO,2.0,TOKYO CEMENT COMPANY (LANKA) PLC,"[b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'...",BMS-10544-LC/00
4,Materials,1.641926e+09,REXP,2.0,RICHARD PIERIS EXPORTS PLC,"[b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'...",BMS-10544-LC/00


In [25]:
items_value_counts = train_df['STOCKCODE'].value_counts()

max_pop = items_value_counts.max()
min_pop = items_value_counts.min()

items_value_counts = items_value_counts.apply(lambda x: np.round((x-min_pop)*100/(max_pop-min_pop), 2))
items_pop_dict = dict(items_value_counts.items())
# items_pop_dict

In [26]:
item_pop_srs = np.round((train_df['STOCKCODE'].value_counts()/train_df['CDSACCNO'].nunique())*100, 2)
items_pop_map = dict(item_pop_srs.items())
# items_pop_map

In [27]:
user_port_pop_map = dict(train_df.groupby('CDSACCNO').apply(lambda x: np.round(sum([items_pop_dict[item] for item in x['STOCKCODE'].values]), 2)).items())
# user_port_pop_map

C:\Users\naradaw\AppData\Local\Temp\ipykernel_27316\1908667837.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_port_pop_map = dict(train_df.groupby('CDSACCNO').apply(lambda x: np.round(sum([items_pop_dict[item] for item in x['STOCKCODE'].values]), 2)).items())


# Results

In [28]:
results_file_path = r"D:\dev work\recommender systems\Atrad_CARS\results\retriever_v3_port_v2__fixed_max_port_size_50_&_tf_listwise_ranking_2024_05_27_11_20_results_fixed_port_50.csv"
results_df = pd.read_csv(results_file_path)
results_df['recommendations'] = results_df["recommendations"].apply(lambda x: ast.literal_eval(x))
results_df.head()

,CDSACCNO,USER_ID,precision@k,recall@k,num_test_items,portfolio_size,recommendations
0,BMS-10544-LC/00,[b'COCO' b'EMER' b'SAMP' b'NDB' b'EML' b'RIL' ...,0.0,0.0,5,23,"[RAL, CWM, DPL, SLND, IDL, ASCO, SHOT, ECL, MA..."
1,BMS-11214-LC/00,[b'LIOC' b'HELA' b'AAIC' b'MELS' b'NTB' b'MASK...,0.0,0.0,5,45,"[CARE, KAHA, DPL, LHCL, LDEV, KGAL, RHTL, LLUB..."
2,BMS-11807-LC/00,[b'BIL' b'AEL' b'RCL' b'HAYL' b'VONE' b'TJL' b...,0.1,0.2,5,17,"[CFIN, TYRE, ABAN, AHPL, CONN, TILE, HHL, HEXP..."
3,BMS-11829-LI/00,[b'SCAP' b'UBC' b'RICH' b'LWL' b'CFVF' b'COMB'...,0.1,0.2,5,21,"[EBCR, UAL, CITH, BLUE, SHL, SAMP, CERA, SOY, ..."
4,BMS-12282-LI/00,[b'EXPO' b'DIAL' b'VONE' b'TKYO' b'PLR' b'EDEN...,0.0,0.0,5,24,"[MSL, ALHP, LCBF, MHDL, MFL, CSF, ASCO, BLI, B..."


In [29]:
results_df['portfolio_size'].max(), results_df['portfolio_size'].min()

portfolios_buckets = np.linspace(
    results_df['portfolio_size'].max(),
    1,
    11
    )

portfolios_buckets = np.sort(portfolios_buckets)
portfolios_buckets

labels = [idx for idx in range(1,11)]
labels

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [30]:
results_df['portfolio_size_bucket'] = pd.cut(
    results_df['portfolio_size'],
    bins = portfolios_buckets,
    labels = labels
)
results_df.head(3)

,CDSACCNO,USER_ID,precision@k,recall@k,num_test_items,portfolio_size,recommendations,portfolio_size_bucket
0,BMS-10544-LC/00,[b'COCO' b'EMER' b'SAMP' b'NDB' b'EML' b'RIL' ...,0.0,0.0,5,23,"[RAL, CWM, DPL, SLND, IDL, ASCO, SHOT, ECL, MA...",5
1,BMS-11214-LC/00,[b'LIOC' b'HELA' b'AAIC' b'MELS' b'NTB' b'MASK...,0.0,0.0,5,45,"[CARE, KAHA, DPL, LHCL, LDEV, KGAL, RHTL, LLUB...",10
2,BMS-11807-LC/00,[b'BIL' b'AEL' b'RCL' b'HAYL' b'VONE' b'TJL' b...,0.1,0.2,5,17,"[CFIN, TYRE, ABAN, AHPL, CONN, TILE, HHL, HEXP...",4


In [31]:
items_pop_dict['SCAP']

67.29

In [32]:
results_df['recommendation_popularity'] = results_df['recommendations'].apply(lambda x: np.round( sum([items_pop_dict[item] for item in x]) ,2))
results_df['portfolio_item_popularity'] = results_df['CDSACCNO'].apply(lambda x: user_port_pop_map[x])
results_df.head(3)

,CDSACCNO,USER_ID,precision@k,recall@k,num_test_items,portfolio_size,recommendations,portfolio_size_bucket,recommendation_popularity,portfolio_item_popularity
0,BMS-10544-LC/00,[b'COCO' b'EMER' b'SAMP' b'NDB' b'EML' b'RIL' ...,0.0,0.0,5,23,"[RAL, CWM, DPL, SLND, IDL, ASCO, SHOT, ECL, MA...",5,177.13,942.28
1,BMS-11214-LC/00,[b'LIOC' b'HELA' b'AAIC' b'MELS' b'NTB' b'MASK...,0.0,0.0,5,45,"[CARE, KAHA, DPL, LHCL, LDEV, KGAL, RHTL, LLUB...",10,206.64,1641.99
2,BMS-11807-LC/00,[b'BIL' b'AEL' b'RCL' b'HAYL' b'VONE' b'TJL' b...,0.1,0.2,5,17,"[CFIN, TYRE, ABAN, AHPL, CONN, TILE, HHL, HEXP...",4,230.34,879.21


In [33]:
results_df['recall@k'].value_counts()

recall@k
0.00    1894
0.20     834
0.40     221
0.60      30
0.80       3
1.00       1
0.25       1
Name: count, dtype: int64

In [34]:
results_df.CDSACCNO.nunique() , len(results_df)

(2984, 2984)

In [35]:
results_df[results_df['recall@k'] != 0].shape[0] *100 / results_df.CDSACCNO.nunique()

36.52815013404826

# EDA

In [36]:
fig = px.histogram(results_df, x="portfolio_size_bucket")

fig.update_layout(width=600, height=400, bargap=0.2)
fig.show()

In [37]:
results_df.head(1)

,CDSACCNO,USER_ID,precision@k,recall@k,num_test_items,portfolio_size,recommendations,portfolio_size_bucket,recommendation_popularity,portfolio_item_popularity
0,BMS-10544-LC/00,[b'COCO' b'EMER' b'SAMP' b'NDB' b'EML' b'RIL' ...,0.0,0.0,5,23,"[RAL, CWM, DPL, SLND, IDL, ASCO, SHOT, ECL, MA...",5,177.13,942.28


In [38]:
fig = px.scatter(results_df, x="precision@k", y="recall@k", color="portfolio_size_bucket") #symbol
fig.update_traces(marker=dict(size=10))
fig.show()

In [39]:
fig = px.scatter(results_df, x="portfolio_item_popularity", y="recommendation_popularity", color="precision@k") #symbol
fig.update_traces(marker=dict(size=10))
fig.show()

In [40]:
fig = px.scatter(results_df, x="portfolio_item_popularity", y="recommendation_popularity", color="recall@k") #symbol
fig.update_traces(marker=dict(size=10))
fig.show()

In [41]:
results_df.head(1)

,CDSACCNO,USER_ID,precision@k,recall@k,num_test_items,portfolio_size,recommendations,portfolio_size_bucket,recommendation_popularity,portfolio_item_popularity
0,BMS-10544-LC/00,[b'COCO' b'EMER' b'SAMP' b'NDB' b'EML' b'RIL' ...,0.0,0.0,5,23,"[RAL, CWM, DPL, SLND, IDL, ASCO, SHOT, ECL, MA...",5,177.13,942.28


In [42]:
fig = px.scatter(
    results_df, 
    x="portfolio_size_bucket", 
    y="portfolio_item_popularity", 
    color = "precision@k",
    # size = "precision@k",
    size_max = 30
    ) #symbol #, color="portfolio_size_bucket"
fig.update_traces(marker=dict(size=15))
fig.show()

In [43]:
fig = px.scatter(
    results_df, 
    x="portfolio_size", 
    y="portfolio_item_popularity", 
    color = "precision@k",
    # size = "precision@k",
    # size_max = 30
    ) #symbol #, color="portfolio_size_bucket"
fig.update_traces(marker=dict(size=15))
fig.show()

In [53]:
fig = px.scatter(
    results_df, 
    x="portfolio_size_bucket", 
    y="portfolio_item_popularity", 
    color = "recall@k",
    # size = "precision@k",
    size_max = 30
    ) #symbol #, color="portfolio_size_bucket"
fig.update_traces(marker=dict(size=15))
fig.show()

In [54]:
fig = px.scatter(
    results_df, 
    x="portfolio_size", 
    y="portfolio_item_popularity", 
    color = "recall@k",
    # size = "precision@k",
    size_max = 30
    ) #symbol #, color="portfolio_size_bucket"
fig.update_traces(marker=dict(size=15))
fig.show()

In [49]:
results_df.head(1)

,CDSACCNO,USER_ID,precision@k,recall@k,num_test_items,portfolio_size,recommendations,portfolio_size_bucket,recommendation_popularity,portfolio_item_popularity
0,BMS-10544-LC/00,[b'COCO' b'EMER' b'SAMP' b'NDB' b'EML' b'RIL' ...,0.0,0.0,5,23,"[RAL, CWM, DPL, SLND, IDL, ASCO, SHOT, ECL, MA...",5,177.13,942.28


In [52]:
fig = px.scatter(
    results_df, 
    x="portfolio_size", 
    y="recall@k", 
    color = "precision@k",
    # size = "precision@k",
    size_max = 30
    ) #symbol #, color="portfolio_size_bucket"
fig.update_traces(marker=dict(size=15))
fig.show()

## item popularity analysis


In [46]:
items_pop_srs[40:55]

NameError: name 'items_pop_srs' is not defined

In [ ]:
item_pop_df = item_pop_srs.to_frame().reset_index()
item_pop_df[:20]

,STOCKCODE,count
0,BIL,55.42
1,EXPO,49.10
2,LOFC,43.72
3,LIOC,41.97
4,RCL,36.40
5,HAYL,34.96
6,SCAP,33.64
7,SAMP,32.98
8,VONE,31.07
9,DIPD,29.43


In [47]:
import plotly.express as px
df = px.data.tips()
fig = px.box(item_pop_df, y='count', hover_data = 'STOCKCODE')
fig.update_layout(width=700, height=400)
fig.show()

NameError: name 'item_pop_df' is not defined